In [81]:
from fp2mp_eval.utils import read_json

results = read_json('data/evaluations.json')

In [82]:
from fp2mp_eval.models import Evaluation
from fp2mp_eval import FP2MPEval
import pingouin as pg

for result in results:
    evaluations = [Evaluation(**e) for e in result['evaluations']]
    result['evaluations_df'] = FP2MPEval.evaluations_to_df(evaluations)
    result['evaluations_long_df'] = FP2MPEval.evaluations_to_long_df(evaluations)
    result['icc_df'] = pg.intraclass_corr(result['evaluations_long_df'], targets='indicator', raters='judge', ratings='score')
    result['icc'] = {}
    for _,icc_row in result['icc_df'].iterrows():
        icc_type = icc_row['Type']
        icc_value = icc_row['ICC']
        result['icc'][icc_type] = icc_value
    result['score'] = result['evaluations_df'].mean().mean()

In [83]:
import pandas as pd

problems = {r['problem'] for r in results}
models = {r['model'] for r in results}
icc_types = set(results[0]['icc'].keys())

icc_dfs = {icc_type : pd.DataFrame(0.0, index=list(problems), columns=list(models)) for icc_type in icc_types}
for result in results:
    problem = result['problem']
    model = result['model']
    for icc_type in icc_types:
        icc_dfs[icc_type].loc[problem, model] = result['icc'][icc_type]

# for result in results:

In [84]:
icc_types

{'ICC(1,1)', 'ICC(1,k)', 'ICC(A,1)', 'ICC(A,k)', 'ICC(C,1)', 'ICC(C,k)'}

In [85]:
icc_dfs['ICC(1,1)']

,meta-llama/llama-3.1-8b-instruct,deepseek/deepseek-v4-pro,openai/gpt-4o-mini,deepseek/deepseek-r1
"Город Кемерово. Создай бренд города, основываясь на локальной идентичности.",0.855154,0.864458,0.697032,0.895131
"Город Санкт-Петербург. Создай бренд города, основываясь на локальной идентичности.",0.845008,0.866019,0.900407,0.718734
Город Санкт-Петербург. Разработай план уплотнения городского центра.,0.837209,0.681818,0.857492,0.747375
Город Кемерово. Разработай план уплотнения городского центра.,0.970711,0.945610,0.549924,0.642857


In [86]:
data = []

for result in results:
    problem = result['problem']
    model = result['model']
    evaluations_df = result['evaluations_df']
    for judge in evaluations_df.index:
        for indicator in evaluations_df.columns:
            data.append({
                'problem': problem,
                'model': model,
                'judge': judge,
                'indicator': indicator,
                'value': evaluations_df.loc[judge, indicator]
            })

data_df = pd.DataFrame(data)
data_df

,problem,model,judge,indicator,value
0,Город Санкт-Петербург. Разработай план уплотне...,deepseek/deepseek-v4-pro,0,framing,4
1,Город Санкт-Петербург. Разработай план уплотне...,deepseek/deepseek-v4-pro,0,decomposition,4
2,Город Санкт-Петербург. Разработай план уплотне...,deepseek/deepseek-v4-pro,0,diversity,3
3,Город Санкт-Петербург. Разработай план уплотне...,deepseek/deepseek-v4-pro,0,coherence,4
4,Город Санкт-Петербург. Разработай план уплотне...,deepseek/deepseek-v4-pro,0,justification,3
...,...,...,...,...,...
635,"Город Кемерово. Создай бренд города, основывая...",meta-llama/llama-3.1-8b-instruct,4,coherence,4
636,"Город Кемерово. Создай бренд города, основывая...",meta-llama/llama-3.1-8b-instruct,4,justification,3
637,"Город Кемерово. Создай бренд города, основывая...",meta-llama/llama-3.1-8b-instruct,4,uncertainty_handling,2
638,"Город Кемерово. Создай бренд города, основывая...",meta-llama/llama-3.1-8b-instruct,4,knowledge_integration,3


In [87]:
data_df.groupby(['model', 'indicator']).agg(
        mean=("value", "mean"),
        std=("value", "std"),
    ).unstack(level=0).reorder_levels([1, 0], axis=1).sort_index(axis=1)

model                 deepseek/deepseek-r1           deepseek/deepseek-v4-pro  \
                                      mean       std                     mean   
indicator                                                                       
coherence                             4.05  0.223607                     4.05   
decomposition                         4.05  0.223607                     4.05   
diversity                             2.75  0.716350                     2.45   
framing                               4.00  0.561951                     3.85   
justification                         3.20  0.615587                     3.15   
knowledge_integration                 4.05  0.686333                     4.20   
metacognition                         2.35  0.587143                     2.30   
uncertainty_handling                  2.35  0.489360                     2.30   

model                           meta-llama/llama-3.1-8b-instruct            \
                            std                             mean       std   
indicator                                                                    
coherence              0.223607                             3.40  0.502625   
decomposition          0.394034                             3.15  0.366348   
diversity              0.604805                             2.00  0.000000   
framing                0.489360                             3.00  0.000000   
justification          0.366348                             2.25  0.444262   
knowledge_integration  0.410391                             3.00  0.324443   
metacognition          0.470162                             1.10  0.307794   
uncertainty_handling   0.470162                             1.50  0.512989   

model                 openai/gpt-4o-mini            
                                    mean       std  
indicator                                           
coherence                           3.50  0.512989  
decomposition                       3.10  0.307794  
diversity                           2.15  0.366348  
framing                             3.00  0.324443  
justification                       2.15  0.366348  
knowledge_integration               3.05  0.223607  
metacognition                       1.60  0.598243  
uncertainty_handling                1.85  0.670820